In [1]:
import pandas as pd
import numpy as np

def read_excel(file_name):
    df = pd.read_excel(file_name)
    return df

def read_txt(file_name):
    file = open(file_name)
    lines = file.readlines()
    return(lines[0])

In [2]:
import os
import glob

def get_files(subfolder, extension):
    dir = f"{os.getcwd()}/content/{subfolder}/"
    tables = glob.glob(f"{dir}*.{extension}")
    return tables

In [3]:
class Analizer:
    def __init__(self, boundary):
        self.results = get_files(subfolder="results", extension="xlsx")
        self.results_df = pd.DataFrame()
        self.boundary = boundary
    
    def has_minimum_requirements(self, df, sort_by="r2"):
        sorted_df = df.sort_values(by=sort_by, ascending=False)
        top_r2 = sorted_df.head(1)[sort_by].values[0]
        if top_r2 < self.boundary:
            return False
        return True
    
    def concatenate_df(self, df, architecture):
        if self.has_minimum_requirements(df):
            df['Architecture'] = architecture
            df = df.rename(columns={'Unnamed: 0': 'model'})
            self.results_df = pd.concat([self.results_df, df], ignore_index=True) 

    def create_results_df(self):
        for file in self.results:
            df = read_excel(file)
            architecture = read_txt(file.replace(".xlsx", ".txt"))
            self.concatenate_df(df, architecture)
        self.results_df = self.results_df.sort_values(by="r2", ascending=False, ignore_index=True)

    def discard_below_average(self, sort_by):
        column_mean = self.results_df[sort_by].mean()      
        self.results_df = self.results_df[self.results_df[sort_by] >= column_mean]
    
    def discard_high_standard_deviation(self):
        r2_val, r2_test = self.results_df['r2_val'], self.results_df['r2_test']
        std_devs = np.abs(r2_val - r2_test)
        mean_std_dev = std_devs.mean()
        self.results_df = self.results_df[std_devs < mean_std_dev]

    def clean_folder(self, subfolder, extension, remove_last=True):
        files = get_files(subfolder, extension)
        models = self.results_df["model"]
        if (remove_last):
            models = models.apply(lambda x: '_'.join(x.rsplit('_', 1)[:-1]))
        for file in files:
            file_name = os.path.basename(file).split('.')[0]
            file_parts = file_name.split('_')            
            dataset_model = f"model_{file_parts[1]}_{file_parts[2]}" 
            if (remove_last == False):
                dataset_model = (f"{dataset_model}_{file_parts[3]}")
            if dataset_model not in models.values:
                os.remove(file)   
        
    def Analize(self):
        self.create_results_df()
        self.discard_below_average(sort_by="r2")
        self.discard_below_average(sort_by="r2_vt")
        self.discard_high_standard_deviation()
        self.results_df.to_excel(f"better_results.xlsx", index=True)
        display(self.results_df)


In [5]:
analize = Analizer(0.9)
analize.Analize()
analize.clean_folder(subfolder="dataset", extension="pkl")
analize.clean_folder(subfolder="results", extension="xlsx")
analize.clean_folder(subfolder="results", extension="txt")
analize.clean_folder(subfolder="models", extension="keras", remove_last=False)



,model,r2,r2_sup,r2_test,r2_val,r2_vt,mse,mse_sup,mse_test,mse_val,mse_vt,mape,rmse,r2_adj,rsd,aic,bic,Architecture
0,model_2_8_21,0.999868,0.959399,0.999681,0.999823,0.999770,0.000131,0.040253,0.000341,0.000213,0.000277,0.023077,0.011440,1.000056,0.011927,179.882538,278.611479,"Hidden Size=[8, 4], regularizer=0.5, learning_..."
1,model_2_8_22,0.999868,0.959385,0.999671,0.999808,0.999759,0.000131,0.040267,0.000352,0.000231,0.000291,0.023034,0.011459,1.000056,0.011947,179.875757,278.604698,"Hidden Size=[8, 4], regularizer=0.5, learning_..."
2,model_2_8_20,0.999868,0.959412,0.999688,0.999838,0.999781,0.000131,0.040240,0.000334,0.000195,0.000264,0.023122,0.011461,1.000056,0.011949,179.875325,278.604267,"Hidden Size=[8, 4], regularizer=0.5, learning_..."
3,model_2_8_23,0.999866,0.959371,0.999659,0.999793,0.999746,0.000133,0.040281,0.000365,0.000249,0.000307,0.022991,0.011513,1.000056,0.012003,179.857098,278.586040,"Hidden Size=[8, 4], regularizer=0.5, learning_..."
4,model_2_8_19,0.999866,0.959424,0.999692,0.999853,0.999790,0.000133,0.040228,0.000329,0.000177,0.000253,0.023168,0.011528,1.000056,0.012019,179.851982,278.580924,"Hidden Size=[8, 4], regularizer=0.5, learning_..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1035,model_17_3_0,0.996657,0.954128,0.999987,0.999779,0.999944,0.003314,0.045479,0.000012,0.000064,0.000038,0.076246,0.057568,1.001042,0.060019,213.419160,336.525618,"Hidden Size=[9, 5], regularizer=0.5, learning_..."
1039,model_13_7_8,0.996639,0.952502,0.996766,0.996090,0.996473,0.003332,0.047091,0.002316,0.002326,0.002321,0.080574,0.057727,1.001136,0.060184,201.408132,317.201335,"Hidden Size=[6, 8], regularizer=0.5, learning_..."
1044,model_4_9_0,0.996625,0.963662,0.996188,0.994979,0.995564,0.003346,0.036027,0.003288,0.004819,0.004054,0.043071,0.057847,1.001421,0.060309,173.399827,272.128769,"Hidden Size=[8, 4], regularizer=0.03, learning..."
1065,model_8_7_7,0.996512,0.951977,0.998996,0.998731,0.998919,0.003458,0.047612,0.000794,0.001395,0.001094,0.093120,0.058806,1.001087,0.061309,213.334068,336.440526,"Hidden Size=[8, 6], regularizer=0.03, learning..."
